# E0 follow-up: is the warp shortfall the sampler or the data?

## Description

**Question.** In the E0 pilot (notebook 01), the Bayes learner kept only 10–30% of the known-warp headroom on `warp` pairs (G), against 50–65% on `window` pairs. Either A's few points really don't pin a 4-parameter-per-axis warp down (**data**: a real price of inferring T, which an amortized Meta-PFN would pay too), or the reference learner is a poor approximation of the Bayes learner (**sampler**: E0 under-reports the headroom). The pilot's own diagnostics point at the sampler: the importance weights over 128/256 prior draws collapse (median ESS 1–3 at n_A ≥ 4) and two independent hypothesis sets disagree by 0.2–0.5 nats per pair, against the 0.025 that E0's rule requires. The pair prior's observation noise is tiny (sd ≈ 1e-3 on [0, 1]), so the likelihood of A's points is sharp and prior draws rarely land in the posterior. **How much of the gap between the E0 Bayes learner and the known-T oracle closes when the posterior over T is sampled properly?**

**Setup.**
- **Pairs**: the first 16 pilot pairs (split 0, σ = 0) of each cell d ∈ {1, 2} × kind ∈ {window, warp}. Same pairs, same PFN (`pfn_large`), same grid; paired with the pilot.
- **Learners, conditioned on the pair's true kind** (the question is about inferring T, not the kind):
  - `kind_is`: E0's importance sampler restricted to the true kind's particles (the pilot's hypothesis set, so it reproduces the pilot);
  - `kind_smc`: sequential Monte Carlo over T (`smc.py`): 256 particles in an unconstrained parameterization whose prior is exactly the pair prior; A's points enter one at a time with adaptive tempering (ESS kept ≥ 128), systematic resampling and 3 random-walk Metropolis moves per resampling, the proposal shaped by the particle cloud. Same target as `kind_is`, the PFN's chain-rule likelihood.
  - `kind_smc_rep`: the same with an independent seed (the SMC learner's own Monte-Carlo error).
  - `oracle_T` (T known) and `lower` (A alone), as in E0.
- Reported alongside: `bayes_smc`, the full E0 learner (mixture over unrelated / identity / window / warp) with the SMC family evidences, against the pilot's `bayes_is`; and the posterior RMS error of T over the box.

**What to look for.** Per cell and n_A ∈ {2, 4, 8}:
- closed = (gain_smc − gain_is) / (gain_oracle − gain_is), the share of the gap that was the sampler, with a 95% pair-bootstrap CI;
- G_smc = gain_smc / gain_oracle: the share of the headroom that survives inferring T, measured properly;
- the posterior T error against n_A: if it falls to ≈ 0 while G_smc stays low, small T errors are expensive under near-noise-free observations (still "data").

**Decision rule (fixed before the run; `analysis/sampler_check.py::verdict`).** Per cell:
- **IMPRECISE** if the SMC learner's MC error exceeds 0.10 nats;
- **SAMPLER** if closed ≥ 0.5 and its CI lower bound > 0;
- **DATA** if closed < 0.2;
- **MIXED** otherwise.

The answer is the majority label over the six warp cells (d × n_A); without a majority, MIXED.
Consequences: SAMPLER → E0's confirm uses the SMC learner (and E0's pilot G numbers are lower bounds); DATA → the warp shortfall is real and is reported as the price of inferring T; IMPRECISE → more particles or moves first.
- *Validity check*: `kind_is` and `bayes_is` must reproduce the pilot's numbers on the same pairs (same code path, same hypothesis seed).

**Limits.**
- σ = 0 only. At σ > 0 neither learner is the Bayes learner (it would need a model of how f_A deviates from f_B; deferred, `research_log/deferred.md`).
- 16 pairs per cell: enough to tell 0.2 from 0.5 when the per-pair effects are large (pilot: gaps of ≈ 2 nats), not for small differences.
- The SMC learner is itself approximate; the replicate bounds its Monte-Carlo error, not its bias (e.g. a missed posterior mode both replicates share).


**Naming (2026-10-22).** The relatedness knob called σ in the description above is `rho` (ρ) in code and configs from 2026-10-22 on; σ is reserved for observation noise. Runs launched before then record it as `task.sigma`; the cells below map it to `rho`. The description itself is pre-registered and left as written.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from meta_pfn.analysis import load_snapshot, show_header
from meta_pfn.analysis.sampler_check import shares, verdict

SNAPSHOT = "results/sampler_check/<sweep_id>"  # set after `just snapshot`
PILOT = "results/headroom/<pilot_sweep_id>"
snap = load_snapshot(SNAPSHOT)
show_header(snap)

## Results

Nats per held-out A query; gains relative to A alone. CIs: 95% pair bootstrap (16 pairs per cell).

In [ ]:
cells = snap.runs.set_index("member")[["task.d", "task.kind"]].set_axis(["d", "kind"], axis=1)
def with_cell(s, name):
    return s.table(name).join(s.runs.set_index("member")[["task.d", "task.kind"]].set_axis(["d", "kind"], axis=1), on="member")
nll, post, smc_runs = (with_cell(snap, n) for n in ("nll", "posterior", "smc"))
s = shares(nll)
s.loc[s.n_a.isin([2, 4, 8])].set_index(["d", "kind", "n_a"]).round(3)

### Validity: the importance sampler reproduces the pilot

In [ ]:
pilot = load_snapshot(PILOT)
pc = pilot.runs.set_index("member").rename(columns={"task.sigma": "task.rho"})[["task.d", "task.kind", "task.rho"]].set_axis(["d", "kind", "rho"], axis=1)
p = pilot.table("nll").join(pc, on="member").query("rho == 0 and predictor == 'bayes' and kind in ['window', 'warp']")
m = nll.query("predictor == 'bayes_is'").merge(p, on=["d", "kind", "seed", "n_a"], suffixes=("", "_pilot"))
print(len(m), "pair x n_A rows matched; max |difference| =", float((m.nll - m.nll_pilot).abs().max()))

### Gain over A alone: importance sampling vs SMC vs T known

Solid: SMC; dashed: importance sampling (E0 pilot); dotted: T known. Shaded: 95% CI of the SMC gain.

In [ ]:
from meta_pfn.analysis.stats import bootstrap_ci
w = nll.pivot_table(index=["d", "kind", "pair", "n_a"], columns="predictor", values="nll").reset_index()
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8), sharey=True)
for ax, ((d, kind), h) in zip(axes, w.groupby(["d", "kind"])):
    for col, name, ls in (("kind_smc", "SMC", "-o"), ("kind_is", "importance sampling", "--o"), ("oracle_T", "T known", ":")):
        g = h.assign(gain=h.lower - h[col]).groupby("n_a").gain
        ax.plot(g.mean().index, g.mean(), ls, ms=3, label=name, c={"kind_smc": "C0", "kind_is": "C1", "oracle_T": "k"}[col])
    ci = np.array([bootstrap_ci((x.lower - x.kind_smc).to_numpy(), n_boot=2000) for _, x in h.groupby("n_a")])
    ax.fill_between(sorted(h.n_a.unique()), ci[:, 0], ci[:, 1], color="C0", alpha=0.15)
    ax.axhline(0, c="k", lw=0.8); ax.set_xscale("symlog", linthresh=1); ax.set_xticks([0, 1, 2, 4, 8], [0, 1, 2, 4, 8])
    ax.set_title(f"d = {d}, {kind}"); ax.set_xlabel("n_A"); ax.grid(alpha=0.3)
axes[0].set_ylabel("gain over A alone (nats/query)"); axes[0].legend(fontsize=8); fig.tight_layout()

### Does A pin T down? Posterior RMS error of T over the box (SMC, both replicates)

In [ ]:
te = post.groupby(["d", "kind", "n_a"])[["t_err", "t_err_rep"]].median()
display(te.unstack("n_a").round(3))
display(post.groupby(["d", "kind", "n_a"])[["p_unrelated", "p_identity", "p_window", "p_warp"]].mean().loc[(slice(None), slice(None), [2, 8]), :].round(2))
smc_runs.groupby(["d", "kind", "family"])[["seconds", "accept", "stages"]].median().round(2)

## Observations

_Filled from the outputs above._

In [ ]:
res = verdict(s)
res

## Verdict

_From the rule above; recorded with `just decide`._